# CFP_Calibration_Efficiency_Frontier

Single-panel calibration-efficiency frontier (Figure 3) showing
raw-to-corrected trajectory for four representative forecasters on
the coverage-width plane at alpha = 0.01.

**Output:** `fig_frontier_killer.pdf`/`.png`

In [ ]:
"""
Calibration-efficiency frontier (Figure 3).
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Self-contained plotting style — duplicated across figure Quantlets
# by Q² convention.
C_GRAY = '#666666'
C_TEAL = '#00A651'
C_RED = '#E31E24'
C_BLUE = '#0066CC'
C_PURPLE = '#7B2FBE'

plt.rcParams.update({
    'font.family': 'serif', 'axes.grid': False,
    'savefig.transparent': True, 'savefig.dpi': 300,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10,
})

SCRIPT_DIR = Path('.').resolve()
BASE = SCRIPT_DIR.parent
DATA = BASE / 'cfp_ijf_data' / 'paper_outputs' / 'tables'
FIG_DIR = BASE / 'figures'
OUT = SCRIPT_DIR

ALPHA = 0.01

SHOW = ['Chronos-Small', 'TimesFM-2.5', 'Lag-Llama', 'GJR-GARCH']
COLORS = {
    'Chronos-Small': '#8b0000',
    'TimesFM-2.5':   '#c41e3a',
    'Lag-Llama':      '#6a0dad',
    'GJR-GARCH':      '#1f77b4',
}
DISPLAY = {
    'Chronos-Small': 'Chronos-Small',
    'TimesFM-2.5':   'TimesFM 2.5',
    'Lag-Llama':      'Lag-Llama',
    'GJR-GARCH':      'GJR-GARCH',
}

df = pd.read_csv(DATA / 'all_results.csv')
d01 = df[df['alpha'] == ALPHA].copy()

data = {}
for model in SHOW:
    mdf = d01[d01['model'] == model]
    data[model] = {
        'raw_width':  mdf['raw_width'].mean(),
        'raw_cov':    1 - mdf['pihat_raw'].mean(),
        'corr_width': mdf['VaR_width'].mean(),
        'corr_cov':   1 - mdf['pihat_cp'].mean(),
    }

fig, ax = plt.subplots(figsize=(8, 5))

for name in SHOW:
    d = data[name]
    c = COLORS[name]
    if name == 'Chronos-Small':
        ax.plot(d['corr_width'], d['corr_cov'], 'o',
                color=c, ms=11, zorder=5)
        ax.annotate(DISPLAY[name],
                    (d['corr_width'], d['corr_cov']),
                    xytext=(12, -5), textcoords='offset points',
                    fontsize=10, fontweight='bold', color=c)
    else:
        ax.plot(d['raw_width'], d['raw_cov'], 'o',
                mfc='white', mec=c, ms=11, mew=2, zorder=5)
        ax.plot(d['corr_width'], d['corr_cov'], 'o',
                color=c, ms=11, zorder=5)
        ax.annotate('',
                    xy=(d['corr_width'], d['corr_cov']),
                    xytext=(d['raw_width'], d['raw_cov']),
                    arrowprops=dict(arrowstyle='->', lw=2, color=c))

ax.annotate(DISPLAY['TimesFM-2.5'],
            (data['TimesFM-2.5']['corr_width'],
             data['TimesFM-2.5']['corr_cov']),
            xytext=(-15, 10), textcoords='offset points',
            fontsize=10, fontweight='bold',
            color=COLORS['TimesFM-2.5'])
ax.annotate(DISPLAY['Lag-Llama'],
            (data['Lag-Llama']['corr_width'],
             data['Lag-Llama']['corr_cov']),
            xytext=(12, 5), textcoords='offset points',
            fontsize=10, fontweight='bold',
            color=COLORS['Lag-Llama'])
ax.annotate(DISPLAY['GJR-GARCH'],
            (data['GJR-GARCH']['raw_width'],
             data['GJR-GARCH']['raw_cov']),
            xytext=(10, 3), textcoords='offset points',
            fontsize=10, fontweight='bold',
            color=COLORS['GJR-GARCH'])

ax.set_title(r'Calibration-Efficiency Frontier ($\alpha = 0.01$)',
             fontsize=12, fontweight='bold')
ax.axhline(y=0.99, color='grey', ls='--', lw=1, zorder=1)
ax.set_xlabel(r'Mean VaR width (narrower $\rightarrow$ more efficient)',
              fontsize=11)
ax.set_ylabel(r'Empirical coverage $1 - \hat{\pi}$', fontsize=11)

all_widths = [data[m]['raw_width'] for m in SHOW] + \
             [data[m]['corr_width'] for m in SHOW]
w_min = min(w for w in all_widths if w > 0.01)
w_max = max(all_widths)
margin = (w_max - w_min) * 0.15
ax.set_xlim(w_min - margin, w_max + margin)
ax.set_ylim(0.963, 1.001)

ax.text(ax.get_xlim()[0] + 0.0005, 0.9905, '99% target',
        fontsize=9, color='grey', style='italic')

hollow = plt.Line2D([0], [0], marker='o', color='grey',
                     mfc='white', ms=9, mew=2, ls='')
filled = plt.Line2D([0], [0], marker='o', color='grey', ms=9, ls='')
ax.legend([hollow, filled], ['Raw', 'Corrected'],
          loc='upper center', bbox_to_anchor=(0.5, -0.1),
          ncol=2, fontsize=9, frameon=False)

axins = ax.inset_axes([0.02, 0.02, 0.25, 0.35])
d = data['Chronos-Small']
c = COLORS['Chronos-Small']
axins.plot(d['raw_width'], d['raw_cov'], 'o',
           mfc='white', mec=c, ms=7, mew=1.5)
axins.plot(d['corr_width'], d['corr_cov'], 'o', color=c, ms=7)
axins.annotate('',
               xy=(d['corr_width'], d['corr_cov']),
               xytext=(d['raw_width'], d['raw_cov']),
               arrowprops=dict(arrowstyle='->', lw=1.5, color=c))
axins.axhline(y=0.99, color='grey', ls='--', lw=0.8)
axins.set_xlim(0.0, 0.05)
axins.set_ylim(0.55, 1.02)
axins.text(0.5, 0.97, 'Chronos-Small\n(full scale)',
           transform=axins.transAxes, fontsize=7,
           color=c, fontweight='bold', ha='center', va='top')
axins.tick_params(labelsize=6)
axins.spines['top'].set_visible(False)
axins.spines['right'].set_visible(False)

plt.tight_layout()
FIG_DIR.mkdir(exist_ok=True)
for ext in ['pdf', 'png']:
    fig.savefig(OUT / f'fig_frontier_killer.{ext}',
               dpi=300, bbox_inches='tight')
    fig.savefig(FIG_DIR / f'fig_frontier_killer.{ext}',
               dpi=300, bbox_inches='tight')

plt.show()
plt.close(fig)
print('Saved: fig_frontier_killer.pdf/.png')